## Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [2]:
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = ChatGroq(model="openai/gpt-oss-20b")
response = model.invoke("Write me an essay on AI")
response

AIMessage(content='**Artificial Intelligence: From Concept to Catalyst**\n\n*Introduction*\n\nArtificial Intelligence (AI) has moved from the realm of speculative fiction into the very fabric of contemporary life. Once a purely theoretical construct, AI now powers everyday conveniences—from voice‑activated assistants to recommendation engines—and is increasingly shaping critical sectors such as healthcare, finance, transportation, and national security. Yet, as AI’s influence expands, so do the questions about its societal impact, ethical boundaries, and long‑term sustainability. This essay traces AI’s evolution, examines its current applications, evaluates the benefits and risks, and looks ahead to the challenges and opportunities that lie ahead.\n\n---\n\n### 1. Defining Artificial Intelligence\n\nAt its core, AI is the study and creation of systems that can perform tasks that normally require human intelligence. These tasks include reasoning, learning from data, perceiving the envir

In [6]:
## Tools
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the weather location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("Whats the weather in Bangalore")
print(response)

content='' additional_kwargs={'reasoning_content': 'We need to call the get_weather function with location "Bangalore".', 'tool_calls': [{'id': 'fc_92cebade-d745-4796-bac5-8a6ea15ae65b', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 123, 'total_tokens': 162, 'completion_time': 0.040813603, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.005864823, 'prompt_tokens_details': None, 'queue_time': 0.047472767, 'total_time': 0.046678426}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0a530-8100-73b1-9847-b5ba6d271ea7-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'fc_92cebade-d745-4796-bac5-8a6ea15ae65b', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata

In [9]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Bangalore'},
  'id': 'fc_92cebade-d745-4796-bac5-8a6ea15ae65b',
  'type': 'tool_call'}]

## Tool Execution Loop

In [10]:
# Step 1 : Model generates tool calls
messages = [{"role":"user","content":"What's the weather in Bangalore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass the results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It’s sunny in Bangalore! 🌞 If you’re heading out, it’s a great day to enjoy the city.


In [11]:
messages

[{'role': 'user', 'content': "What's the weather in Bangalore?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Bangalore?" We need to call get_weather function.', 'tool_calls': [{'id': 'fc_b31418a2-cf4f-4eb0-a9c0-2f37e4d25c60', 'function': {'arguments': '{"location":"Bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 124, 'total_tokens': 167, 'completion_time': 0.046548273, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.00605874, 'prompt_tokens_details': None, 'queue_time': 0.399436327, 'total_time': 0.052607013}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5979a0e1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a53b-1908-7893-b084-4904c3c6d84e-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': 'fc_

In [12]:
model_with_tools

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001FBCE5F3A60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FBCE5F3F40>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather location', 'parameters': {'properties': {'loc